#### Notebook do laboratório de visualização de dados

 Dataset utilizado: [dados brapci](../datasets/Brapci-2019-03-26.html) disponibilizados em https://cedap.ufrgs.br/xmlui/handle/20.500.11959/1189

In [89]:
%pip install numpy pandas altair tomli

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [90]:
import tomllib
import pandas as pd
import altair as alt
from pathlib import Path
from IPython.display import HTML, display
import uuid

In [91]:
toml_path = Path("../datasets/Brapci-2019-03-26.toml")

with toml_path.open("rb") as arquivo:
    dados = tomllib.load(arquivo)

df = pd.DataFrame(dados["record"])
#df.head()

In [92]:
#df.info()

In [93]:
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["page"] = pd.to_numeric(df["page"], errors="coerce")

df[["year", "page"]].describe()

,year,page
count,24570.000000,25620.000000
mean,2007.192145,47886.333958
std,11.328941,27028.735116
min,1920.000000,119.000000
25%,2003.000000,26088.250000
50%,2011.000000,40106.500000
75%,2015.000000,72305.000000
max,2020.000000,99945.000000


In [94]:
df.describe(include="all")

,author,title,source,edition,year,section,keywords,abstract,page,link
count,25621,25621,25621,25621,24570.000000,25621,25621,25621,25620.000000,25621
unique,14850,21232,73,11751,NaN,408,11948,20178,NaN,25621
top,,,Pesquisa Brasileira em Ciência da Informação e...,,NaN,Artigo,,,NaN,http://www.brapci.inf.br/index.php/res/v/119
freq,745,614,3791,614,NaN,5590,12734,5071,NaN,1
mean,NaN,NaN,NaN,NaN,2007.192145,NaN,NaN,NaN,47886.333958,NaN
std,NaN,NaN,NaN,NaN,11.328941,NaN,NaN,NaN,27028.735116,NaN
min,NaN,NaN,NaN,NaN,1920.000000,NaN,NaN,NaN,119.000000,NaN
25%,NaN,NaN,NaN,NaN,2003.000000,NaN,NaN,NaN,26088.250000,NaN
50%,NaN,NaN,NaN,NaN,2011.000000,NaN,NaN,NaN,40106.500000,NaN
75%,NaN,NaN,NaN,NaN,2015.000000,NaN,NaN,NaN,72305.000000,NaN


##### Gráfico interativo de estatisticas gerais


In [95]:
import re
import unicodedata

# A coluna "section" tem 408 valores diferentes: o mesmo tipo aparece escrito de vários jeitos
# ("Artigo", "Artigo Científico", "Artículo", "Investigacione", ...). Estas regras juntam em tipos.
# São testadas em ordem e a primeira que casar define o grupo; o que não casar vira "Outras".
# Os padrões (regex) são comparados com o nome da seção sem acentos e em minúsculas.
GRUPOS_SECAO = [
    ("Resumo de artigo",              r"resumo de artigo"),
    ("Tese, dissertação e TCC",       r"tese|dissert|monografia|conclusao de curso|trabalho academico|discente"),
    ("Resenha",                       r"\bresen|\brecens|book review|revision de libro|novedade bibliografica|livro recebido"),
    ("Artigo de revisão",             r"\brevis(ao|ion|ione)|review article"),
    ("Relato de pesquisa/experiência", r"relato|relatorio|estud(o|io) de caso|reporte de caso|experiencia para divulgar"),
    ("Entrevista",                    r"entrevista|interview"),
    ("Expediente, sumário e normas",  r"expediente|sumario|indice|norma|diretriz|comissao|corpo editorial|avaliador"
                                      r"|agradec|credito|capa|orelha|completa|completo|conteudo desta|pagina inicial"
                                      r"|colaboran|about this issue|abreviatura|glosario|errata"),
    ("Editorial e apresentação",      r"editor|apresentacao|apresentao|presentaci|prefacio|foreword|carta a"),
    ("Ensaio e opinião",              r"ensai|ensay|opini|ponto de vista|debate|comentario|reflexione|carta abierta"),
    ("Notícia",                       r"noticia|notievento|en portada|actualidad|in memori|necrel|obituario|nota de vida|homenagem"),
    ("Trabalho de evento",            r"evento|anais|congresso|palestra|conferencia|ponencia|forum|painel|poster|pecha kucha"
                                      r"|papers eiica|short papers|\bgt ?\d|eixo \d|ebbc|enancib|cbbd|contecsi|fesabid"
                                      r"|jornade|integrar|sndij|premio"),
    ("Dossiê e seção especial",       r"dossi|especial|tematic|suplemento|^parte i"),
    ("Comunicação e pesquisa em andamento", r"comunica|em andamento"),
    # vazio, "Outro", "Sem Secção" e o nome da revista RBBD usado como seção
    ("Sem seção definida",            r"^$|sem sec|^outro$|^tipo$|^rbbd"),
    ("Artigo",                        r"artigo|articulo|article|original|investigaci|papers|contribucione corta"),
]


def sem_acento(texto):
    return "".join(c for c in unicodedata.normalize("NFKD", texto) if not unicodedata.combining(c))


def grupo_da_secao(secao):
    nome = sem_acento(secao).lower().strip()
    for grupo, padrao in GRUPOS_SECAO:
        if re.search(padrao, nome):
            return grupo
    return "Outras"

In [96]:
N_RESUMO = 15   # quantas fontes aparecem antes de clicar em "mostrar mais"
N_ANOS = 16     # quantas faixas de ano (as mais recentes) aparecem antes de "mostrar mais"

por_fonte = (
    df.groupby("source")
      .size()
      .reset_index(name="trabalhos")
      .rename(columns={"source": "categoria"})
      .sort_values("trabalhos", ascending=False)
      .assign(grafico="Por fonte")
)
por_fonte["ordem"] = range(len(por_fonte))  # maior contagem no topo
total_fontes = len(por_fonte)

# Seções: são 408 valores diferentes (variações de "Artigo", em português e espanhol,
# nomes de evento etc.). GRUPOS_SECAO / grupo_da_secao (célula anterior) juntam em tipos.
df["tipo_secao"] = df["section"].map(grupo_da_secao)
por_secao = (
    df["tipo_secao"].value_counts()
      .rename_axis("categoria")
      .reset_index(name="trabalhos")
)
# maior contagem no topo; "Outras" e "Sem seção definida" sempre no fim
por_secao["no_fim"] = por_secao["categoria"].map({"Sem seção definida": 1, "Outras": 2}).fillna(0)
por_secao = por_secao.sort_values(["no_fim", "trabalhos"], ascending=[True, False]).drop(columns="no_fim")
por_secao["ordem"] = range(len(por_secao))
por_secao["grafico"] = "Por seção"

# Anos agrupados de 3 em 3, terminando no ano mais recente (ex.: 2018–2020, 2015–2017, ...)
anos = df["year"].dropna().astype(int)
ano_max, ano_min = anos.max(), anos.min()
inicio = (ano_max - 2) - 3 * ((ano_max - anos) // 3)
inicios = range(ano_max - 2, ano_min - 3, -3)  # todas as faixas, mesmo as vazias
por_ano = (
    inicio.value_counts()
      .reindex(inicios, fill_value=0)
      .rename_axis("inicio")
      .reset_index(name="trabalhos")
      .assign(
          categoria=lambda x: x["inicio"].astype(str) + "–" + (x["inicio"] + 2).astype(str),
          grafico="Por ano",
      )
      .drop(columns="inicio")
)
por_ano["ordem"] = range(len(por_ano))  # mais recente no topo
# período coberto pelas N_ANOS faixas mostradas antes de "mostrar mais" (para o título)
periodo_resumido = f"{ano_max - 3 * N_ANOS + 1}–{ano_max}"

# A coluna "page" é o id do registro (igual ao do link), não o nº de páginas.
# O nº de páginas vem do intervalo em "edition": "n. 2, v. 13, p. 31-41" -> 11 páginas
intervalo = df["edition"].str.extract(r"p\.\s*(\d+)\s*-\s*(\d+)").astype(float)
df["paginas"] = intervalo[1] - intervalo[0] + 1
df.loc[df["paginas"] <= 0, "paginas"] = None  # descarta intervalos inválidos (ex.: "p. 2014-1")

# Faixas de 5 em 5 páginas (0–4, 5–9, ..., 40–44); 45 ou mais fica tudo em "45+"
limites = list(range(0, 46, 5)) + [float("inf")]
rotulos = [f"{i}–{i + 4} págs." for i in range(0, 45, 5)] + ["45+ págs."]

por_paginas = (
    df.dropna(subset=["paginas"])
      .assign(categoria=lambda x: pd.cut(x["paginas"], bins=limites, labels=rotulos, right=False))
      .groupby("categoria", observed=False)
      .size()
      .reset_index(name="trabalhos")
      .assign(grafico="Por nº de páginas")
)
por_paginas["categoria"] = por_paginas["categoria"].astype(str)
por_paginas["ordem"] = range(len(por_paginas) - 1, -1, -1)  # 0–4 embaixo, 45+ no topo

dados_graficos = pd.concat([por_fonte, por_secao, por_ano, por_paginas], ignore_index=True)
opcoes = list(dados_graficos["grafico"].unique())

AZUL = "#4c78a8"

# Seletor: botões de rádio do Vega (estilizados com CSS logo abaixo).
# Obs.: botões desenhados dentro do gráfico com selection_point também funcionam,
# mas aí o Vega não encolhe o gráfico ao trocar e sobra espaço em branco embaixo.
seletor = alt.param(
    name="grafico_escolhido",
    value=opcoes[0],
    bind=alt.binding_radio(options=opcoes, name="Trabalhos"),
)
# "Mostrar mais" de fontes e anos: um checkbox (o CSS transforma em botão)
mostrar_todas = alt.param(
    name="mostrar_todas",
    value=False,
    bind=alt.binding_checkbox(name="Mostrar mais"),
)

LARGURA_ROTULOS = 230

titulo = (
    f"'Trabalhos ' + lower({seletor.name})"
    f" + ({seletor.name} == 'Por fonte' && !{mostrar_todas.name} ? ' ({N_RESUMO} maiores de {total_fontes})' : '')"
    f" + ({seletor.name} == 'Por ano' && !{mostrar_todas.name} ? ' ({periodo_resumido})' : '')"
    f" + ({seletor.name} == 'Por seção' ? ' (agrupadas por tipo)' : '')"
)
# quantas barras cada gráfico mostra antes de "mostrar mais" (os outros mostram tudo)
limites_resumo = {"Por fonte": N_RESUMO, "Por ano": N_ANOS}
escondida = " || ".join(
    f"(datum.grafico == '{g}' && datum.ordem >= {n})" for g, n in limites_resumo.items()
)

chart = (
    alt.Chart(dados_graficos)
    .mark_bar(color=AZUL, cornerRadiusEnd=3)
    .encode(
        x=alt.X("trabalhos:Q", title="Número de trabalhos"),
        y=alt.Y(
            "categoria:N",
            sort=alt.EncodingSortField("ordem"),
            title=None,
            # largura fixa para os rótulos: o gráfico e o título não "andam" ao trocar
            axis=alt.Axis(minExtent=LARGURA_ROTULOS, maxExtent=LARGURA_ROTULOS, labelLimit=LARGURA_ROTULOS - 10),
        ),
        tooltip=[
            alt.Tooltip("categoria:N", title="Categoria"),
            alt.Tooltip("trabalhos:Q", title="Trabalhos", format=","),
        ],
    )
    .add_params(seletor, mostrar_todas)
    .transform_filter(alt.datum.grafico == seletor)
    # sem "mostrar mais", fontes e anos mostram só as primeiras barras
    .transform_filter(f"{mostrar_todas.name} || !({escondida})")
    .properties(
        width=700,
        height=alt.Step(18),  # altura se ajusta ao nº de barras do gráfico escolhido
        title=alt.Title(alt.expr(titulo)),
        # recalcula o tamanho a cada troca (senão fica a altura do primeiro gráfico)
        autosize=alt.AutoSizeParams(type="pad", resize=True),
    )
    .configure(background="white")
    .configure_view(stroke=None)
    .configure_axis(labelFontSize=11, titleFontSize=12, titleColor="#555", gridColor="#eee")
    .configure_title(fontSize=16, anchor="start", offset=12)
)

# Estilo dos controles. O Vega gera:
#   rádio:    <div class="vega-bind"><span class="vega-bind-name">..</span><span class="vega-bind-radio"><label><input type="radio">Texto</label>...</span></div>
#   checkbox: <div class="vega-bind"><label><span class="vega-bind-name">..</span><input type="checkbox"></label></div>
estilo = """
<style>
/* fundo branco no bloco todo (no tema escuro do editor a saída do notebook é preta) */
.graficos-brapci {
  display: inline-block; background: #fff; color: #222;
  padding: 16px 20px; border-radius: 10px;
}
.graficos-brapci .chart-wrapper { display: flex; flex-direction: column; }
.graficos-brapci .vega-bindings { display: contents; }
.graficos-brapci .vega-bind { display: flex; align-items: center; gap: 10px; font-family: system-ui, sans-serif; }
.graficos-brapci .vega-bind-name { font-size: 13px; font-weight: 600; color: #555; }
.graficos-brapci .vega-bind input { position: absolute; opacity: 0; pointer-events: none; }

/* seletor de gráfico: acima do gráfico */
.graficos-brapci .vega-bind:has(.vega-bind-radio) { order: -1; margin: 0 0 16px; }
.graficos-brapci .vega-bind-radio {
  display: inline-flex; gap: 4px; padding: 4px;
  background: #eef1f5; border-radius: 999px;
}
.graficos-brapci .vega-bind-radio label {
  padding: 6px 16px; border-radius: 999px; cursor: pointer;
  font-size: 13px; color: #555; transition: background .15s, color .15s;
}
.graficos-brapci .vega-bind-radio label:hover { background: #dde3ea; color: #222; }
.graficos-brapci .vega-bind-radio label:has(input:checked) {
  background: #4c78a8; color: #fff; box-shadow: 0 1px 3px rgba(0,0,0,.2);
}

/* "mostrar mais": abaixo do gráfico, alinhado com as barras, só em fontes e anos */
.graficos-brapci .vega-bind:has(input[type=checkbox]) { order: 1; margin: 8px 0 0 MARGEMpx; }
.graficos-brapci:not(:has(input[value="Por fonte"]:checked, input[value="Por ano"]:checked))
  .vega-bind:has(input[type=checkbox]) { display: none; }
.graficos-brapci .vega-bind label:has(input[type=checkbox]) {
  padding: 6px 16px; border-radius: 999px; cursor: pointer;
  font-size: 13px; color: #4c78a8; font-weight: 600; background: #eef1f5;
}
.graficos-brapci .vega-bind label:has(input[type=checkbox]):hover { background: #dde3ea; }
.graficos-brapci .vega-bind label:has(input[type=checkbox]) .vega-bind-name { display: none; }
/* texto do botão depende do gráfico escolhido e de estar aberto (checked) ou não */
.graficos-brapci:has(input[value="Por fonte"]:checked) label:has(input[type=checkbox])::after {
  content: "Mostrar todas as TOTAL_FONTES fontes ▾";
}
.graficos-brapci:has(input[value="Por fonte"]:checked) label:has(input[type=checkbox]:checked)::after {
  content: "Mostrar só as RESUMO maiores ▴";
}
.graficos-brapci:has(input[value="Por ano"]:checked) label:has(input[type=checkbox])::after {
  content: "Mostrar anos anteriores ▾";
}
.graficos-brapci:has(input[value="Por ano"]:checked) label:has(input[type=checkbox]:checked)::after {
  content: "Mostrar só PERIODO ▴";
}
</style>
"""
estilo = (
    estilo.replace("MARGEM", str(LARGURA_ROTULOS + 5))
          .replace("TOTAL_FONTES", str(total_fontes))
          .replace("RESUMO", str(N_RESUMO))
          .replace("PERIODO", periodo_resumido)
)

# template="universal" espera o Vega carregar antes de desenhar (o padrão falha
# dentro do notebook) e o id único evita conflito com outras saídas
grafico_html = chart.to_html(fullhtml=False, template="universal", output_div=f"vis-{uuid.uuid4().hex}")
display(HTML(estilo + '<div class="graficos-brapci">' + grafico_html + "</div>"))

##### Palavras-chave mais comuns por fonte

As três fontes com mais trabalhos. Clique para trocar de fonte.

In [97]:
import json

N_KEYWORDS = 15  # quantas palavras-chave mostrar por fonte

# fonte (como está no dataset) -> nome curto para o botão
FONTES_KEYWORDS = {
    "Pesquisa Brasileira em Ciência da Informação e Biblioteconomia": "PBCIB",
    "Ciência da Informação": "Ciência da Informação",
    "Perspectivas em Ciência da Informação": "Perspectivas em CI",
}

# Uma linha por (trabalho, palavra-chave). As palavras vêm separadas por ";".
kw = (
    df[df["source"].isin(FONTES_KEYWORDS)]
      .assign(keyword=lambda x: x["keywords"].str.split(";"))
      .explode("keyword")
      .assign(keyword=lambda x: x["keyword"].str.strip())
      .query("keyword != ''")
)
# junta grafias que só mudam em maiúsculas/minúsculas e conta cada trabalho uma vez
kw["chave"] = kw["keyword"].str.lower()
kw = kw.drop_duplicates(subset=["link", "chave"])
# para exibir, usa a grafia mais comum de cada palavra-chave (evitando a toda em MAIÚSCULAS)
def escolher_grafia(formas):
    contagem = formas.value_counts()
    normais = [f for f in contagem.index if not f.isupper()]
    return normais[0] if normais else contagem.index[0].capitalize()

grafia = kw.groupby("chave")["keyword"].agg(escolher_grafia)

por_keyword = (
    kw.groupby(["source", "chave"])
      .size()
      .reset_index(name="trabalhos")
      .sort_values(["source", "trabalhos"], ascending=[True, False])
      .groupby("source")
      .head(N_KEYWORDS)
      .assign(
          categoria=lambda x: x["chave"].map(grafia),
          grafico=lambda x: x["source"].map(FONTES_KEYWORDS),
      )
)
por_keyword["ordem"] = por_keyword.groupby("source").cumcount()  # mais comum no topo

# quantos trabalhos de cada fonte têm palavras-chave (para o título)
com_kw = kw.groupby("source")["link"].nunique()
total = df["source"].value_counts()
# título de cada fonte: [nome completo, contagem] (duas linhas)
descricao = {
    curto: [
        f"Palavras-chave: {fonte}",
        f"{com_kw[fonte]:,} de {total[fonte]:,} trabalhos têm palavras-chave".replace(",", "."),
    ]
    for fonte, curto in FONTES_KEYWORDS.items()
}

opcoes_kw = list(FONTES_KEYWORDS.values())
seletor_kw = alt.param(
    name="fonte_escolhida",
    value=opcoes_kw[0],
    bind=alt.binding_radio(options=opcoes_kw, name="Fonte"),
)
# título = descrição da fonte escolhida (procura pelo nome curto)
lista_descricoes = json.dumps(descricao, ensure_ascii=False)

chart_kw = (
    alt.Chart(por_keyword)
    .mark_bar(color=AZUL, cornerRadiusEnd=3)
    .encode(
        x=alt.X("trabalhos:Q", title="Número de trabalhos com a palavra-chave"),
        y=alt.Y(
            "categoria:N",
            sort=alt.EncodingSortField("ordem"),
            title=None,
            axis=alt.Axis(minExtent=LARGURA_ROTULOS, maxExtent=LARGURA_ROTULOS, labelLimit=LARGURA_ROTULOS - 10),
        ),
        tooltip=[
            alt.Tooltip("categoria:N", title="Palavra-chave"),
            alt.Tooltip("trabalhos:Q", title="Trabalhos", format=","),
        ],
    )
    .add_params(seletor_kw)
    .transform_filter(alt.datum.grafico == seletor_kw)
    .properties(
        width=700,
        height=alt.Step(18),
        title=alt.Title(alt.expr(f"{lista_descricoes}[{seletor_kw.name}]")),
        autosize=alt.AutoSizeParams(type="pad", resize=True),
    )
    .configure(background="white")
    .configure_view(stroke=None)
    .configure_axis(labelFontSize=11, titleFontSize=12, titleColor="#555", gridColor="#eee")
    .configure_title(fontSize=16, anchor="start", subtitleColor="#777", offset=12)
)

# mesmo estilo de botões do gráfico anterior (variável "estilo" da célula de cima)
grafico_html = chart_kw.to_html(fullhtml=False, template="universal", output_div=f"vis-{uuid.uuid4().hex}")
display(HTML(estilo + '<div class="graficos-brapci">' + grafico_html + "</div>"))

##### Temas ao longo do tempo

Porcentagem dos trabalhos (com palavras-chave) de cada ano que citam alguns temas. Clique num tema da legenda para destacar a linha dele.

In [98]:
ANO_INICIO, ANO_FIM = 1995, 2018  # antes de 1995 há poucos trabalhos/ano; 2019 tem só 2 (dataset é de mar/2019)

# tema -> palavras-chave (em minúsculas) que contam como esse tema
TEMAS = {
    "Internet": ["internet"],
    "Gestão do conhecimento": ["gestão do conhecimento", "knowledge management"],
    "Gestão da informação": ["gestão da informação", "information management"],
    "Bibliometria": ["bibliometria", "bibliometrics", "bibliometría"],
    "Acesso aberto": ["acesso aberto", "acesso livre", "open access", "acceso abierto"],
    "Competência em informação": ["competência informacional", "competência em informação",
                                  "information literacy", "alfabetización informacional"],
    "Preservação digital": ["preservação digital", "digital preservation"],
    "Redes sociais": ["rede social", "redes sociais", "social network", "redes sociales"],
}
palavra_para_tema = {p: tema for tema, palavras in TEMAS.items() for p in palavras}

# só trabalhos com ano e com palavras-chave, no período escolhido
com_kw = df[(df["keywords"].str.strip() != "") & df["year"].between(ANO_INICIO, ANO_FIM)].copy()
com_kw["ano"] = com_kw["year"].astype(int)
trabalhos_por_ano = com_kw.groupby("ano").size()

# uma linha por (trabalho, tema); cada trabalho conta uma vez por tema
temas_trabalho = (
    com_kw.assign(keyword=lambda x: x["keywords"].str.split(";"))
      .explode("keyword")
      .assign(tema=lambda x: x["keyword"].str.strip().str.lower().map(palavra_para_tema))
      .dropna(subset=["tema"])
      .drop_duplicates(subset=["link", "tema"])
)

tendencia = (
    temas_trabalho.groupby(["ano", "tema"]).size()
      .unstack(fill_value=0)
      .reindex(index=range(ANO_INICIO, ANO_FIM + 1), columns=list(TEMAS), fill_value=0)
)
# % dos trabalhos com palavras-chave daquele ano que citam o tema
percentual = tendencia.div(trabalhos_por_ano, axis=0) * 100
# média móvel de 3 anos para suavizar o sobe-e-desce de um ano para o outro
suavizado = percentual.rolling(3, center=True, min_periods=1).mean()

dados_temas = (
    suavizado.stack().rename("percentual").reset_index()
      .merge(tendencia.stack().rename("trabalhos").reset_index(), on=["ano", "tema"])
      .merge(trabalhos_por_ano.rename("total_ano").reset_index(), on="ano")
)

# clicar num tema da legenda destaca a linha dele (clicar de novo / fora volta ao normal)
destaque = alt.selection_point(fields=["tema"], bind="legend")

base = alt.Chart(dados_temas).encode(
    x=alt.X(
        "ano:Q",
        title="Ano",
        scale=alt.Scale(domain=[ANO_INICIO, ANO_FIM], nice=False),
        axis=alt.Axis(format="d", tickMinStep=1),
    ),
    y=alt.Y("percentual:Q", title="% dos trabalhos com palavras-chave"),
    color=alt.Color(
        "tema:N",
        sort=list(TEMAS),
        title="Tema (clique para destacar)",
        scale=alt.Scale(scheme="tableau10"),
        legend=alt.Legend(orient="right", labelFontSize=12, titleFontSize=12, symbolStrokeWidth=4),
    ),
    opacity=alt.when(destaque).then(alt.value(1)).otherwise(alt.value(0.12)),
)

linhas = base.mark_line(strokeWidth=2.5, interpolate="monotone")
pontos = base.mark_circle(size=40).encode(
    tooltip=[
        alt.Tooltip("tema:N", title="Tema"),
        alt.Tooltip("ano:Q", title="Ano"),
        alt.Tooltip("percentual:Q", title="% (média de 3 anos)", format=".1f"),
        alt.Tooltip("trabalhos:Q", title="Trabalhos no ano"),
        alt.Tooltip("total_ano:Q", title="Trabalhos c/ palavras-chave no ano"),
    ],
)

chart_temas = (
    alt.layer(linhas, pontos)
    .add_params(destaque)
    .properties(
        width=700,
        height=380,
        title=alt.Title(
            "Temas ao longo do tempo",
            subtitle=f"% dos trabalhos com palavras-chave que citam cada tema, {ANO_INICIO}–{ANO_FIM} (média móvel de 3 anos)",
        ),
    )
    .configure(background="white")
    .configure_view(stroke=None)
    .configure_axis(labelFontSize=11, titleFontSize=12, titleColor="#555", gridColor="#eee")
    .configure_title(fontSize=16, anchor="start", subtitleColor="#777", offset=12)
)

# mesmo cartão branco dos outros gráficos (variável "estilo" da célula do gráfico geral)
grafico_html = chart_temas.to_html(fullhtml=False, template="universal", output_div=f"vis-{uuid.uuid4().hex}")
display(HTML(estilo + '<div class="graficos-brapci">' + grafico_html + "</div>"))

##### Palavras mais comuns nos títulos

Palavras soltas e grupos de 2, 3 e 4 palavras seguidas que mais aparecem nos títulos. Artigos, preposições e conjunções (o, a, de, da, e, ...) não contam: "gestão do conhecimento" é um grupo de 2 palavras. Singular e plural contam juntos; cada grupo conta uma vez por título e títulos repetidos contam uma vez só.

In [99]:
import re
from collections import Counter

N_NGRAMAS = 20  # quantos grupos de palavras mostrar em cada gráfico

# Palavras "vazias" (artigos, preposições...) em português, espanhol e inglês.
# Elas não entram na contagem: "gestão do conhecimento" conta como o grupo de 2 palavras
# "gestão conhecimento" (o rótulo do gráfico mostra o trecho como está no título).
STOPWORDS = set("""
a à ao aos as às o os um uma uns umas de da das do dos dum duma em na nas no nos num numa
por pela pelas pelo pelos para pra com sem sob sobre entre até após ante contra desde
e ou mas nem que se como quando onde porque qual quais quem cujo cuja
é são ser foi era está estão ter tem há sua seu suas seus seu este esta estes estas esse essa
isso isto aquele aquela não mais menos muito muitos muita muitas também já
el la los las lo un una unos unas del al y o u en con sin por para sobre entre desde hacia
que como cuando donde su sus es son ser este esta estos estas ese esa no más
the of and in on for to a an with from by at as is are its their this that
v n p vol nº ed pp
""".split())  # última linha: "v. 1, n. 2, p. 3" de títulos que citam o fascículo


def limpar_titulo(titulo):
    titulo = re.sub(r"@[A-Za-z]{2}(-[A-Za-z]{2})?\s*$", "", titulo)  # tira "@pt-BR", "@es-ES"...
    return titulo.lower()


def palavras(titulo):
    # só letras (com acento) e hífen interno; números e pontuação separam palavras
    return re.findall(r"[a-zà-öø-ÿ]+(?:-[a-zà-öø-ÿ]+)*", titulo)


# Plural -> singular (português e espanhol), testado em ordem. Só troca se a forma no
# singular também aparece em algum título: "bibliotecas" -> "biblioteca", mas
# "português" continua igual (não existe "portuguê").
REGRAS_PLURAL = [
    ("ões", "ão"), ("ães", "ão"), ("ãos", "ão"), ("ais", "al"), ("éis", "el"), ("óis", "ol"),
    ("ns", "m"), ("res", "r"), ("zes", "z"), ("les", "l"), ("s", ""), ("es", ""),
]


def singular(palavra, vocabulario):
    if palavra in STOPWORDS:
        return palavra
    for sufixo, troca in REGRAS_PLURAL:
        if palavra.endswith(sufixo) and len(palavra) > len(sufixo) + 2:
            candidata = palavra[: -len(sufixo)] + troca
            if candidata in vocabulario:
                return candidata
    return palavra


def ngramas(tokens, normalizados, n):
    """Gera (grupo normalizado, trecho como está no título) para cada sequência de n
    palavras seguidas, pulando artigos/preposições (STOPWORDS)."""
    posicoes = [i for i, p in enumerate(tokens) if p not in STOPWORDS]
    for j in range(len(posicoes) - n + 1):
        janela = posicoes[j:j + n]
        if n == 1 and len(tokens[janela[0]]) < 3:  # palavras soltas muito curtas ("ii"...)
            continue
        chave = " ".join(normalizados[i] for i in janela)
        trecho = " ".join(tokens[janela[0]:janela[-1] + 1])  # inclui o "da", "do"... do meio
        yield chave, trecho


# Títulos repetidos ("Editorial", "Apresentação", "Resenha"...) entram uma vez só,
# senão eles dominam a contagem.
titulos = (
    df["title"].map(limpar_titulo).str.strip()
      .loc[lambda s: s != ""]
      .drop_duplicates()
)
tokens_por_titulo = titulos.map(palavras)
vocabulario = {p for tokens in tokens_por_titulo for p in tokens}
normalizado = {p: singular(p, vocabulario) for p in vocabulario}

linhas = []
for n in range(1, 5):
    contagem = Counter()  # grupo normalizado -> nº de títulos
    formas = {}           # grupo normalizado -> Counter das formas originais (para o rótulo)
    for tokens in tokens_por_titulo:
        norm = [normalizado[p] for p in tokens]
        vistos = set()
        for chave, original in ngramas(tokens, norm, n):
            formas.setdefault(chave, Counter())[original] += 1
            if chave not in vistos:  # cada grupo conta no máximo uma vez por título
                vistos.add(chave)
                contagem[chave] += 1
    rotulo = "1 palavra" if n == 1 else f"{n} palavras"
    for ordem, (chave, qtd) in enumerate(contagem.most_common(N_NGRAMAS)):
        # rótulo = forma mais comum (ex.: "bibliotecas universitárias")
        linhas.append({"grafico": rotulo, "categoria": formas[chave].most_common(1)[0][0],
                       "trabalhos": qtd, "ordem": ordem})
por_ngrama = pd.DataFrame(linhas)

opcoes_ng = list(por_ngrama["grafico"].unique())
seletor_ng = alt.param(
    name="tamanho_grupo",
    value=opcoes_ng[0],
    bind=alt.binding_radio(options=opcoes_ng, name="Grupos de"),
)

chart_ng = (
    alt.Chart(por_ngrama)
    .mark_bar(color=AZUL, cornerRadiusEnd=3)
    .encode(
        x=alt.X("trabalhos:Q", title="Número de títulos"),
        y=alt.Y(
            "categoria:N",
            sort=alt.EncodingSortField("ordem"),
            title=None,
            axis=alt.Axis(minExtent=LARGURA_ROTULOS, maxExtent=LARGURA_ROTULOS, labelLimit=LARGURA_ROTULOS - 10),
        ),
        tooltip=[
            alt.Tooltip("categoria:N", title="Palavras"),
            alt.Tooltip("trabalhos:Q", title="Títulos", format=","),
        ],
    )
    .add_params(seletor_ng)
    .transform_filter(alt.datum.grafico == seletor_ng)
    .properties(
        width=700,
        height=alt.Step(18),
        title=alt.Title(
            alt.expr(
                f"{seletor_ng.name} == '1 palavra' ? 'Palavras mais comuns nos títulos'"
                f" : 'Grupos de ' + {seletor_ng.name} + ' seguidas mais comuns nos títulos'"
            ),
            subtitle=[
                "Artigos, preposições e conjunções (o, a, de, da, do, e, ...) não entram na contagem:",
                "\"gestão do conhecimento\" é um grupo de 2 palavras. Singular e plural contam juntos.",
            ],
        ),
        autosize=alt.AutoSizeParams(type="pad", resize=True),
    )
    .configure(background="white")
    .configure_view(stroke=None)
    .configure_axis(labelFontSize=11, titleFontSize=12, titleColor="#555", gridColor="#eee")
    .configure_title(fontSize=16, anchor="start", subtitleColor="#777", subtitleFontSize=12, offset=12)
)

# mesmo cartão e botões dos outros gráficos (variável "estilo" da célula do gráfico geral)
grafico_html = chart_ng.to_html(fullhtml=False, template="universal", output_div=f"vis-{uuid.uuid4().hex}")
display(HTML(estilo + '<div class="graficos-brapci">' + grafico_html + "</div>"))

##### Frequência e ano médio dos grupos de palavras

Cada ponto é um grupo de 2 palavras dos títulos: no eixo X, em quantos títulos aparece; no eixo Y, o ano médio desses trabalhos (mais alto = termo mais recente).

In [100]:
from collections import defaultdict

MIN_TITULOS = 20  # só grupos que aparecem em pelo menos esse número de títulos

# grupos que ganham rótulo (escritos como no tooltip) -> de que lado do ponto fica o texto
DESTAQUES = {
    "ciência da informação": "acima",
    "bibliotecas universitárias": "abaixo",
    "gestão do conhecimento": "direita",
    "redes sociais": "acima",
    "competência em informação": "direita",
    "big data": "esquerda",
    "mídias sociais": "direita",
    "lei de acesso": "direita",
    "biblioteca nacional": "direita",
    "disseminação seletiva": "direita",
    "automação de bibliotecas": "direita",
    "banco de dados": "abaixo",
}
POSICOES = {  # lado -> como o texto é alinhado/deslocado em relação ao ponto
    "direita": dict(align="left", dx=8, dy=0),
    "esquerda": dict(align="right", dx=-8, dy=0),
    "acima": dict(align="center", dx=0, dy=-13),
    "abaixo": dict(align="center", dx=0, dy=14),
}

# Mesmo processo do gráfico anterior (usa palavras, ngramas e normalizado da célula de cima),
# agora guardando o ano de cada título. Títulos repetidos e sem ano ficam de fora.
titulos_ano = (
    df.assign(titulo=df["title"].map(limpar_titulo).str.strip())
      .query("titulo != ''")
      .dropna(subset=["year"])
      .drop_duplicates(subset="titulo")
)

anos_do_grupo = defaultdict(list)    # grupo normalizado -> anos dos títulos que o usam
formas_do_grupo = defaultdict(Counter)
for titulo, ano in zip(titulos_ano["titulo"], titulos_ano["year"]):
    tokens = palavras(titulo)
    norm = [normalizado.get(p, p) for p in tokens]
    vistos = set()
    for chave, trecho in ngramas(tokens, norm, 2):
        formas_do_grupo[chave][trecho] += 1
        if chave not in vistos:  # conta cada título uma vez
            vistos.add(chave)
            anos_do_grupo[chave].append(int(ano))

dispersao = pd.DataFrame(
    [
        {
            "grupo": formas_do_grupo[chave].most_common(1)[0][0],
            "titulos": len(anos),
            "ano_medio": sum(anos) / len(anos),
            "primeiro": min(anos),
            "ultimo": max(anos),
        }
        for chave, anos in anos_do_grupo.items()
        if len(anos) >= MIN_TITULOS
    ]
)
dispersao["destaque"] = dispersao["grupo"].isin(DESTAQUES)
dispersao["posicao"] = dispersao["grupo"].map(DESTAQUES)

base = alt.Chart(dispersao).encode(
    x=alt.X(
        "titulos:Q",
        title="Títulos que usam o grupo de palavras (escala log)",
        scale=alt.Scale(type="log", domain=[MIN_TITULOS * 0.9, dispersao["titulos"].max() * 2], nice=False),
        axis=alt.Axis(format="d", grid=True, values=[20, 30, 50, 100, 200, 300, 500, 1000]),
    ),
    y=alt.Y(
        "ano_medio:Q",
        title="Ano médio dos trabalhos",
        scale=alt.Scale(zero=False, nice=True),
        axis=alt.Axis(format="d"),
    ),
)

cor_ano = alt.Color(
    "ano_medio:Q",
    title="Ano médio",
    # um tom só (azul), de claro a escuro: antigo = "desbotado", recente = forte.
    # Evita vermelho/verde, que passariam a ideia de "ruim"/"bom".
    scale=alt.Scale(range=["#b3cde3", "#4a90c2", "#08306b"]),
    legend=alt.Legend(format="d", gradientLength=200),
)

pontos = base.mark_circle(size=55, opacity=0.75).encode(
    color=cor_ano,
    tooltip=[
        alt.Tooltip("grupo:N", title="Grupo de palavras"),
        alt.Tooltip("titulos:Q", title="Títulos"),
        alt.Tooltip("ano_medio:Q", title="Ano médio", format=".1f"),
        alt.Tooltip("primeiro:Q", title="Primeiro ano"),
        alt.Tooltip("ultimo:Q", title="Último ano"),
    ],
)
# contorno escuro nos pontos com rótulo
marcados = base.mark_circle(size=70, fill=None, stroke="#333", strokeWidth=1.5).transform_filter(
    alt.datum.destaque
)
# Rótulos: uma camada por lado (direita, esquerda...), cada uma com "halo" branco
# (o mesmo texto com contorno branco por baixo) para dar para ler sobre os pontos.
rotulos = []
for lado, desloc in POSICOES.items():
    estilo_texto = dict(**desloc, baseline="middle", fontSize=12, fontWeight=600)
    so_esse_lado = alt.datum.posicao == lado
    rotulos.append(
        base.mark_text(**estilo_texto, stroke="white", strokeWidth=4, strokeOpacity=0.9)
        .encode(text="grupo:N").transform_filter(so_esse_lado)
    )
    rotulos.append(
        base.mark_text(**estilo_texto, color="#222").encode(text="grupo:N").transform_filter(so_esse_lado)
    )

chart_dispersao = (
    alt.layer(pontos, marcados, *rotulos)
    .properties(
        width=760,
        height=440,
        title=alt.Title(
            "Grupos de 2 palavras nos títulos: frequência e ano médio",
            subtitle=[
                f"Cada ponto é um grupo de 2 palavras que aparece em pelo menos {MIN_TITULOS} títulos. "
                "Quanto mais alto, mais recentes os trabalhos que o usam.",
                "Artigos, preposições e conjunções não contam (\"gestão do conhecimento\" = 2 palavras); "
                "singular e plural contam juntos.",
            ],
        ),
    )
    .configure(background="white")
    .configure_view(stroke=None)
    .configure_axis(labelFontSize=11, titleFontSize=12, titleColor="#555", gridColor="#eee")
    .configure_title(fontSize=16, anchor="start", subtitleColor="#777", subtitleFontSize=12, offset=12)
)

# mesmo cartão branco dos outros gráficos (variável "estilo" da célula do gráfico geral)
grafico_html = chart_dispersao.to_html(fullhtml=False, template="universal", output_div=f"vis-{uuid.uuid4().hex}")
display(HTML(estilo + '<div class="graficos-brapci">' + grafico_html + "</div>"))